# 04 — Benchmark Results

Run the full benchmark and visualize results:
- Recall@5 and nDCG@10 for all 5 retrieval strategies
- Latency comparison
- Bar charts per metric

Strategies: naive → hybrid → reranker → graph → agentic

In [ ]:
import sys
sys.path.insert(0, '..')
from dotenv import load_dotenv
load_dotenv('../.env')

## Run benchmark (sample for speed)

In [ ]:
from src.evaluation.benchmark import run_benchmark

results = run_benchmark(
    strategies=['naive', 'hybrid', 'reranker', 'graph', 'agentic'],
    sample_n=8,
)

## Visualization

In [ ]:
import matplotlib.pyplot as plt

strategies = list(results.keys())
metrics = ['recall@5', 'ndcg@10', 'avg_latency_ms']
colors = ['#4C72B0', '#55A868', '#C44E52', '#8172B2', '#CCB974']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric in zip(axes, metrics):
    values = [results[s].get(metric, 0) for s in strategies]
    bars = ax.bar(strategies, values, color=colors[:len(strategies)])
    ax.set_title(metric)
    ax.set_ylim(0, max(values) * 1.2 if max(values) > 0 else 1)
    ax.tick_params(axis='x', rotation=15)
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('RAG Strategy Comparison — Vietnamese Legal Documents', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../results/benchmark_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved to results/benchmark_comparison.png")

## Deep Agent benchmark (uses orchestrator)

In [ ]:
from src.agents.orchestrator import agent

agent_config = {"configurable": {"thread_id": "nb-benchmark-01"}}
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Run the full benchmark on the 8 sample gold queries using all 5 strategies and give me a comparison table."
    }]
}, config=agent_config)

print(result['messages'][-1].content)